<a href="https://colab.research.google.com/github/aiwithpavansama/Stages-of-learning-LLM-Data-pre-processing-from-Scratch-LLM-Base-/blob/main/Stage_1_Data_preprossing_from_Scratch(LLM_Base).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **#reading the words**

### Stage 1: Data Preparation

In [36]:
!pip install pypdf
from pypdf import PdfReader

In [37]:
# reading the text from the PDF and then getting first 100 characters

reader = PdfReader("/content/drive/MyDrive/The_Verdict.pdf")   # or "/path/to/file.pdf"

raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() or ""

print("total number of characters",len(raw_text))
print(raw_text[:99])

total number of characters 21813
1
The Verdict
Edith Wharton
1908
Exported from Wikisource on May 20, 20242
I HAD always thought Jac


In [38]:
# splitting the words to create tokens

In [39]:
import re
result=re.split(r'(\s)',raw_text)
result = [item for item in result if item.strip()]
print(len(result))
print(result[:99])

3848
['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20,', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius--', 'though', 'a', 'good', 'fellow', 'enough--so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that,', 'in', 'the', 'height', 'of', 'his', 'glory,', 'he', 'had', 'dropped', 'his', 'painting,', 'married', 'a', 'rich', 'widow,', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera.', '(Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence.)', '"The', 'height', 'of', 'his', 'glory"--that', 'was', 'what', 'the', 'women', 'called', 'it.', 'I', 'can', 'hear', 'Mrs.', 'Gideon', 'Thwing--his', 'last', 'Chicago', 'sitter--', 'deploring', 'his', 'unaccountable', 'abdication.', '"Of', 'course']


In [40]:
# we would have created a token on this one but we cannpt since we need camma and other special characters to be as a token so we further split the data so that orginal content is not missed

In [41]:
preprocessed=re.split(r'([,.:;?_!()\']|--|\s)',raw_text)
preprocessed=[item for item in preprocessed if item.strip()]
print(preprocessed[:30])

['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20', ',', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough']


In [42]:
print(len(preprocessed))

4845


# **Step 2: Creating a Token ID's**:

In [43]:
# sorting in order and giving the token id's
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print("total number of unique words",vocab_size)

total number of unique words 1279


In [44]:
vocab={token:Integer for Integer,token in enumerate(all_words)}

In [45]:
for i, item in enumerate(vocab.items()):
  print(item)
  if i>=50:
    break

('!', 0)
('"', 1)
('"10', 2)
('"19', 3)
('"Ah', 4)
('"Be', 5)
('"Begin', 6)
('"By', 7)
('"Come', 8)
('"Destroyed', 9)
('"Don', 10)
('"Gisburns"', 11)
('"Grindles', 12)
('"Hang', 13)
('"Has', 14)
('"How', 15)
('"I', 16)
('"If', 17)
('"It', 18)
('"Jack', 19)
('"Money', 20)
('"Moon-dancers"', 21)
('"Mr', 22)
('"Mrs', 23)
('"My', 24)
('"Never', 25)
('"Of', 26)
('"Oh', 27)
('"Once', 28)
('"Only', 29)
('"Or', 30)
('"That', 31)
('"The', 32)
('"Then', 33)
('"There', 34)
('"This', 35)
('"We', 36)
('"Well', 37)
('"What', 38)
('"When', 39)
('"Why', 40)
('"Yes', 41)
('"You', 42)
('"but', 43)
('"deadening', 44)
('"dragged', 45)
('"effects"', 46)
('"interesting"', 47)
('"lift', 48)
('"obituary"', 49)
('"strongest', 50)


# **#implement Tokenizer class for encode and decode**

In [46]:
import re

# Define the upgraded tokenizer class
class SimpleTokenizer:
    def __init__(self, vocab=None):
        if vocab is not None:
            self.str_to_int = vocab
            self.int_to_str = {idx: token for token, idx in vocab.items()}
        else:
            self.str_to_int = {}
            self.int_to_str = {}

    def encode(self, text):
        if isinstance(text, str):
            # Tokenize on spaces, punctuation, and special tokens
            tokens = re.split(r'([,.:;?_!()\']|--|\s|<\|endoftext\|>|<\|unk\|>)', text)
            tokens = [item for item in tokens if item and item.strip()]
        else:
            tokens = text

        encoded_ids = []
        for token in tokens:
            if token in self.str_to_int:
                encoded_ids.append(self.str_to_int[token])
            elif '<|unk|>' in self.str_to_int:
                encoded_ids.append(self.str_to_int['<|unk|>'])
            else:
                # If <|unk|> is not configured in vocab, skip the unknown token
                pass
        return encoded_ids

    def decode(self, token_ids):
        tokens = [self.int_to_str[idx] for idx in token_ids if idx in self.int_to_str]
        text = " ".join(tokens)
        # Clean up spaces before punctuation marks
        text = re.sub(r"\s+([,.:;?(!'\"])", r"\1", text)
        return text

# Fix: Create vocab_with_specials before using it
all_words_with_specials = sorted(list(set(preprocessed)))
all_words_with_specials.extend(["<|unk|>", "<|endoftext|>"])
vocab_with_specials = {token: integer for integer, token in enumerate(all_words_with_specials)}

# Now initialize the upgraded tokenizer successfully
tokenizer = SimpleTokenizer(vocab=vocab_with_specials)

# Test with 'hello' and '<|endoftext|>'
test_text = "hello It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>"
ids = tokenizer.encode(test_text)
print("Encoded IDs (first ID is <|unk|>):", ids)
print("Decoded output:", tokenizer.decode(ids))

Encoded IDs (first ID is <|unk|>): [1279, 127, 1223, 835, 1150, 1144, 1272, 720, 1125, 56, 683, 1127, 402, 846, 201, 549, 1233, 53, 680, 852, 1127, 157, 56, 701, 1087, 844, 1157, 783, 1157, 1262, 1248, 113, 629, 605, 1193, 664, 876, 58, 147, 963, 56, 701, 956, 1223, 201, 1117, 927, 1280]
Decoded output: <|unk|> It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>


### Upgraded Tokenizer with Special Tokens Support

We will now recreate our vocabulary list and include two special tokens:
* `<|unk|>`: Used to represent any word that was not present in our training corpus (like `'hello'`).
* `<|endoftext|>`: Used to mark boundaries between different documents or sections.

In [47]:
# 1. Create a new sorted list of unique words and append the special tokens
all_words_with_specials = sorted(list(set(preprocessed)))
all_words_with_specials.extend(["<|unk|>", "<|endoftext|>"])

# 2. Build the updated vocabulary mapping
vocab_with_specials = {token: integer for integer, token in enumerate(all_words_with_specials)}

print("New vocab size:", len(vocab_with_specials))
print("ID of <|unk|>:", vocab_with_specials["<|unk|>"])
print("ID of <|endoftext|>:", vocab_with_specials["<|endoftext|>"])

New vocab size: 1281
ID of <|unk|>: 1279
ID of <|endoftext|>: 1280


In [48]:
import re

class SimpleTokenizer:
    def __init__(self, vocab=None):
        if vocab is not None:
            self.str_to_int = vocab
            self.int_to_str = {idx: token for token, idx in vocab.items()}
        else:
            self.str_to_int = {}
            self.int_to_str = {}

    def encode(self, text):
        if isinstance(text, str):
            # Tokenize on spaces, punctuation, and special tokens
            tokens = re.split(r'([,.:;?_!()\']|--|\s|<\|endoftext\|>|<\|unk\|>)', text)
            tokens = [item for item in tokens if item and item.strip()]
        else:
            tokens = text

        encoded_ids = []
        for token in tokens:
            if token in self.str_to_int:
                encoded_ids.append(self.str_to_int[token])
            elif '<|unk|>' in self.str_to_int:
                encoded_ids.append(self.str_to_int['<|unk|>'])
            else:
                # If <|unk|> is not configured in vocab, skip the unknown token
                pass
        return encoded_ids

    def decode(self, token_ids):
        tokens = [self.int_to_str[idx] for idx in token_ids if idx in self.int_to_str]
        text = " ".join(tokens)
        # Clean up spaces before punctuation marks
        text = re.sub(r"\s+([,.:;?(!'\"])", r"\1", text)
        return text

# Initialize the upgraded tokenizer using the vocabulary with special tokens
tokenizer = SimpleTokenizer(vocab=vocab_with_specials)

In [49]:
# Test with 'hello' and '<|endoftext|>'
test_text = "hello It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>"
ids = tokenizer.encode(test_text)
print("Encoded IDs (first ID is <|unk|>):", ids)
print("Decoded output:", tokenizer.decode(ids))

Encoded IDs (first ID is <|unk|>): [1279, 127, 1223, 835, 1150, 1144, 1272, 720, 1125, 56, 683, 1127, 402, 846, 201, 549, 1233, 53, 680, 852, 1127, 157, 56, 701, 1087, 844, 1157, 783, 1157, 1262, 1248, 113, 629, 605, 1193, 664, 876, 58, 147, 963, 56, 701, 956, 1223, 201, 1117, 927, 1280]
Decoded output: <|unk|> It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>


In [50]:
# 1. Create a new sorted list of unique words and append the special tokens
all_words_with_specials = sorted(list(set(preprocessed)))
all_words_with_specials.extend(["<|unk|>", "<|endoftext|>"])

# 2. Build the updated vocabulary mapping
vocab_with_specials = {token: integer for integer, token in enumerate(all_words_with_specials)}

print("New vocab size:", len(vocab_with_specials))
print("ID of <|unk|>:", vocab_with_specials["<|unk|>"])
print("ID of <|endoftext|>:", vocab_with_specials["<|endoftext|>"])

New vocab size: 1281
ID of <|unk|>: 1279
ID of <|endoftext|>: 1280


In [51]:
tokenizer=SimpleTokenizer(vocab)

In [52]:
text =""" Pavan Kalyan sama It was not till three years later that, in the course of a few
weeks' idling on the Riviera, it suddenly occurred to me to
wonder why Gisburn had given up his painting. On
reflection, it really was a tempting problem """

In [53]:
ids=tokenizer.encode(text)
print(ids)

[127, 1223, 835, 1150, 1144, 1272, 720, 1125, 56, 683, 1127, 402, 846, 201, 549, 1233, 53, 680, 852, 1127, 157, 56, 701, 1087, 844, 1157, 783, 1157, 1262, 1248, 113, 629, 605, 1193, 664, 876, 58, 147, 963, 56, 701, 956, 1223, 201, 1117, 927]


In [54]:
tokenizer.decode(ids)

"It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem"

### Explanation: Why "Pavan Kalyan" was skipped

The reason no IDs were generated for `Pavan` and `Kalyan` is because the `tokenizer` in cell `3jrFpvtghisD` was initialized with the original `vocab` which doesn't contain `<|unk|>` or those specific names.

If we re-initialize the tokenizer using `vocab_with_specials` (which contains `<|unk|>`), both words will map to the `<|unk|>` token ID (`1279`):

In [55]:
# Re-initialize tokenizer with the vocabulary that contains special tokens
tokenizer_with_specials = SimpleTokenizer(vocab=vocab_with_specials)

# Encode the text containing unseen words
ids_demo = tokenizer_with_specials.encode(text)
print("Encoded IDs (Notice the first two IDs are 1279 for '<|unk|>'):", ids_demo[:5])
print("Decoded text:", tokenizer_with_specials.decode(ids_demo))

Encoded IDs (Notice the first two IDs are 1279 for '<|unk|>'): [1279, 1279, 1279, 127, 1223]
Decoded text: <|unk|> <|unk|> <|unk|> It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem


### 1. Word-Based Tokenizer
* **Meaning**: Splits text into tokens based on individual words. This is typically done using whitespace and punctuation marks as boundaries.
* **Pros**: Highly intuitive; tokens map directly to meaningful concepts/words.
* **Cons**: Leads to a massive vocabulary size. It cannot handle out-of-vocabulary (OOV) words (any unseen word, like spelling mistakes or new words, maps to `<|unk|>`).

### 2. Character-Based Tokenizer
* **Meaning**: Splits text into individual characters (letters, numbers, and symbols).
* **Pros**: Tiny vocabulary size (usually just the alphabet and punctuation) and zero Out-Of-Vocabulary (OOV) words since any word can be spelled out.
* **Cons**: The sequence length becomes extremely long (each word contains multiple tokens), making it harder for models to learn long-range relationships between words.

### 3. Byte Pair Encoding (BPE) Tokenizer
* **Meaning**: A subword-level tokenizer. It starts at the character level and iteratively merges the most frequently co-occurring pairs of characters/subwords into new tokens until a target vocabulary size is met.
* **Pros**: Strikes a perfect balance. It keeps common words whole (e.g., `"the"`), but breaks unknown or rare words down into meaningful subword units (e.g., `"unbelievable"` -> `["un", "believ", "able"]`). It completely avoids the OOV problem without inflating sequence lengths.

### Why We Use `tiktoken`

While we can build simple character or word-based tokenizers from scratch (like our `SimpleTokenizer`), modern Large Language Models (LLMs) like GPT-4, GPT-3.5, and others use advanced subword tokenization algorithms. **`tiktoken`** is a fast Byte Pair Encoding (BPE) tokenizer developed by OpenAI.

Here is why we use it:

1. **High Performance & Speed**: `tiktoken` is written in Rust, making it extremely fast compared to pure Python tokenizers. It can handle massive text corpora very quickly.
2. **No Out-Of-Vocabulary (OOV) Errors**: Instead of dropping or replacing unknown words (like "Pavan Kalyan" mapping to `<|unk|>`), it breaks unknown or rare words down into smaller, recognizable subword units or bytes.
3. **Exact Alignment with OpenAI Models**: If you are working with GPT models, `tiktoken` guarantees that the tokens and IDs match the exact token counts and representations used by the model APIs. This is crucial for precise cost estimation and managing context limits.

In [56]:
!pip3 install tiktoken

In [57]:
import importlib
import tiktoken

In [58]:
tokenizer=tiktoken.get_encoding("gpt2")

**The line tokenizer = tiktoken.get_encoding("gpt2")**
 initializes the Byte Pair Encoding (BPE) tokenizer configuration specifically trained for OpenAI's GPT-2 model family. Here is exactly what this line does:

Loads the GPT-2 Vocabulary: It fetches the pre-trained dictionary of 50,257 unique tokens that GPT-2 was trained on.
Retrieves BPE Merging Rules: It configures the rules used to group individual characters or bytes together into high-frequency subword patterns.
Instantiates an Encoder Object: It returns a Python object equipped with optimized methods like .encode() (to turn text into a list of integers) and .decode() (to reconstruct the original text from token integers)

In [59]:
text_to_encode=vocab_with_specials

In [60]:
# Fix the error by explicitly setting text_to_encode to our target text string first
text_to_encode = "peter parker is one of the best super hero  in the marvel series"

gpt_ids = tokenizer.encode(text_to_encode)
print("Token IDs:", gpt_ids)

Token IDs: [79, 2357, 3952, 263, 318, 530, 286, 262, 1266, 2208, 4293, 220, 287, 262, 22672, 2168]


In [61]:
gpt_decode_text=tokenizer.decode(gpt_ids)
print(gpt_decode_text)

peter parker is one of the best super hero  in the marvel series


### Visualizing Byte Pair Encoding (BPE) for `text_to_encode`

To see BPE in action, we can take each token ID generated by `tiktoken` and decode it individually. This shows us exactly where the tokenizer splits words (including spaces, which are treated as part of the tokens):

In [62]:
# We use the GPT-2 tokenizer we initialized earlier
gpt_ids = tokenizer.encode(text_to_encode)

print(f"Original Text: \"{text_to_encode}\"\n")
print("--- BPE Token Breakdown ---")
print(f"{'Token ID':<10} | {'Token (Decoded String)':<25} | {'Notes'}")
print("-" * 60)

for token_id in gpt_ids:
    decoded_piece = tokenizer.decode([token_id])
    # Help visualize spaces by replacing them with a visible character
    visible_piece = decoded_piece.replace(" ", "•")

    note = "Whole word" if decoded_piece.strip() == decoded_piece else "Subword / Includes Space"
    print(f"{token_id:<10} | {repr(visible_piece):<25} | {note}")

Original Text: "peter parker is one of the best super hero  in the marvel series"

--- BPE Token Breakdown ---
Token ID   | Token (Decoded String)    | Notes
------------------------------------------------------------
79         | 'p'                       | Whole word
2357       | 'eter'                    | Whole word
3952       | '•park'                   | Subword / Includes Space
263        | 'er'                      | Whole word
318        | '•is'                     | Subword / Includes Space
530        | '•one'                    | Subword / Includes Space
286        | '•of'                     | Subword / Includes Space
262        | '•the'                    | Subword / Includes Space
1266       | '•best'                   | Subword / Includes Space
2208       | '•super'                  | Subword / Includes Space
4293       | '•hero'                   | Subword / Includes Space
220        | '•'                       | Subword / Includes Space
287        | '•in'             

# Creating input traget Pairs

In [63]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    # tokenize the entire text
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    # use a sliding window to chunk the book into overlapping sequences of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    # Return the pre-constructed tensors directly to avoid redundant tensor creation warnings
    return self.input_ids[idx], self.target_ids[idx]

### How the Sliding Window Creates Input-Output Pairs

To train an LLM to predict the next token in a sequence, we use a **sliding window** approach. This technique splits a continuous stream of token IDs into multiple training examples consisting of an input sequence ($X$) and its corresponding shifted target sequence ($Y$).

#### Key Parameters:
* **`max_length` (Context Window)**: The number of tokens in each training sample's input chunk.
* **`stride`**: The step size of our sliding window. A stride of `1` means the window shifts forward by exactly one token to create the next example.

#### Visual Demonstration:

Imagine we have a tokenized sequence: `[50, 51, 52, 53, 54, 55, 56]` and set `max_length = 4`, `stride = 1`:

* **Step 1 (Index 0)**:
  * **Input (X)**: `[50, 51, 52, 53]`
  * **Target (Y)**: `[51, 52, 53, 54]` (shifted right by 1)

* **Step 2 (Index 1 - shifted by 1)**:
  * **Input (X)**: `[51, 52, 53, 54]`
  * **Target (Y)**: `[52, 53, 54, 55]`

* **Step 3 (Index 2 - shifted by another 1)**:
  * **Input (X)**: `[52, 53, 54, 55]`
  * **Target (Y)**: `[53, 54, 55, 56]`

This overlapping sequence generation allows the model to learn what token comes next from multiple positions within the same context chunk, maximizing the efficiency of our training data.

In [64]:
# 1. Convert the vocabulary keys from vocab_with_specials dictionary back into a single continuous text string
vocab_text = " ".join(list(vocab_with_specials.keys()))

# 2. Instantiate the TextDataset using the vocabulary string with a sliding window of max_length=4 and stride=1
vocab_dataset = TextDataset(vocab_text, tokenizer, max_length=4, stride=1)

print(f"Total training examples generated from vocab_with_specials: {len(vocab_dataset)}\n")

# 3. Print the first 5 input-target pairs to verify
for i in range(min(5, len(vocab_dataset))):
    x, y = vocab_dataset[i]
    print(f"--- Batch / Pair {i+1} ---")
    print(f"Input (X) IDs:   {x.tolist()}")
    print(f"Target (Y) IDs:  {y.tolist()}")
    print(f"Input tokens:    {[tokenizer.decode([token]) for token in x.tolist()]}")
    print(f"Target tokens:   {[tokenizer.decode([token]) for token in y.tolist()]}")
    print()

Total training examples generated from vocab_with_specials: 1753

--- Batch / Pair 1 ---
Input (X) IDs:   [0, 366, 366, 940]
Target (Y) IDs:  [366, 366, 940, 366]
Input tokens:    ['!', ' "', ' "', '10']
Target tokens:   [' "', ' "', '10', ' "']

--- Batch / Pair 2 ---
Input (X) IDs:   [366, 366, 940, 366]
Target (Y) IDs:  [366, 940, 366, 1129]
Input tokens:    [' "', ' "', '10', ' "']
Target tokens:   [' "', '10', ' "', '19']

--- Batch / Pair 3 ---
Input (X) IDs:   [366, 940, 366, 1129]
Target (Y) IDs:  [940, 366, 1129, 366]
Input tokens:    [' "', '10', ' "', '19']
Target tokens:   ['10', ' "', '19', ' "']

--- Batch / Pair 4 ---
Input (X) IDs:   [940, 366, 1129, 366]
Target (Y) IDs:  [366, 1129, 366, 10910]
Input tokens:    ['10', ' "', '19', ' "']
Target tokens:   [' "', '19', ' "', 'Ah']

--- Batch / Pair 5 ---
Input (X) IDs:   [366, 1129, 366, 10910]
Target (Y) IDs:  [1129, 366, 10910, 366]
Input tokens:    [' "', '19', ' "', 'Ah']
Target tokens:   ['19', ' "', 'Ah', ' "']



In [65]:
from torch.utils.data import DataLoader

# 1. Initialize the PyTorch DataLoader with batch size 4 and shuffling enabled
batch_size = 4
data_loader = DataLoader(
    vocab_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

# 2. Extract and inspect the first batch
data_iter = iter(data_loader)
first_batch_x, first_batch_y = next(data_iter)

print(f"DataLoader established!")
print(f"Batch Input (X) shape:  {first_batch_x.shape}")
print(f"Batch Target (Y) shape: {first_batch_y.shape}\n")

print("--- Decoded Batch Samples ---")
for i in range(batch_size):
    x_ids = first_batch_x[i].tolist()
    y_ids = first_batch_y[i].tolist()
    print(f"Sample {i+1} in Batch:")
    print(f"  Input text:  {repr(tokenizer.decode(x_ids))}")
    print(f"  Target text: {repr(tokenizer.decode(y_ids))}")
    print()

DataLoader established!
Batch Input (X) shape:  torch.Size([4, 4])
Batch Target (Y) shape: torch.Size([4, 4])

--- Decoded Batch Samples ---
Sample 1 in Batch:
  Input text:  ' frame frames free frequently'
  Target text: ' frames free frequently friend'

Sample 2 in Batch:
  Input text:  ' part passages passing past'
  Target text: ' passages passing past past'

Sample 3 in Batch:
  Input text:  ' object objects occurred oddly'
  Target text: ' objects occurred oddly of'

Sample 4 in Batch:
  Input text:  'organ Among And Are'
  Target text: ' Among And Are Ar'



### What is a Token Embedding?

Once our tokenizer converts raw text into numerical **Token IDs** (integers like `[79, 2357, 3952]`), these integers cannot be used directly by deep learning models for mathematical calculations.

**Token Embedding** is the next step in the pipeline that converts these abstract integer IDs into dense, continuous vectors of real numbers (floats).

#### How it works:

1. **Embedding Matrix**: The model maintains a lookup table (matrix) of size `(vocab_size, embedding_dimension)`. For example, with GPT-2, the vocabulary size is `50,257` and the embedding dimension might be `768` floats per token.
2. **ID as an Index**: When we pass a Token ID (e.g., `79`), the model acts like it is looking up row index `79` in this massive matrix.
3. **Semantic Representation**: Initially, these vectors are randomized. During training, the model adjusts the floating-point values so that tokens with similar meanings, grammatical roles, or contexts end up with vectors that are geometrically close to each other in high-dimensional space (e.g., `'king'` - `'man'` + `'woman'` ≈ `'queen'`).

In [66]:
import torch
import torch.nn as nn

# 1. Define the vocabulary size and embedding dimension
# We use the standard GPT-2 vocabulary size (50,257) which aligns with our tiktoken tokenizer
vocab_size = 50257
embedding_dim = 256  # Map each token ID to a 256-dimensional continuous vector

In [67]:
# 2. Instantiate the PyTorch Embedding Layer
token_embedding_layer = nn.Embedding(vocab_size, embedding_dim)
print(token_embedding_layer)

Embedding(50257, 256)


In [68]:
# 3. Retrieve a single batch from our established DataLoader
data_iter = iter(data_loader)
batch_x, batch_y = next(data_iter)

In [69]:
# 4. Pass the input token IDs through the embedding layer
batch_embeddings_x = token_embedding_layer(batch_x)
batch_embeddings_y = token_embedding_layer(batch_y)

# 5. Inspect the output shapes
print(f"Original Token ID batch shape:       {batch_x.shape} (batch_size, sequence_length)")
print(f"Resulting Input Embeddings shape:    {batch_embeddings_x.shape} (batch_size, sequence_length, embedding_dim)")
print(f"Resulting Target Embeddings shape:   {batch_embeddings_y.shape} (batch_size, sequence_length, embedding_dim)")

Original Token ID batch shape:       torch.Size([4, 4]) (batch_size, sequence_length)
Resulting Input Embeddings shape:    torch.Size([4, 4, 256]) (batch_size, sequence_length, embedding_dim)
Resulting Target Embeddings shape:   torch.Size([4, 4, 256]) (batch_size, sequence_length, embedding_dim)


# Adding Positional Encoding to our Embeddings**
Why We Use Positional Encoding
Although Token Embeddings successfully map words (like "weekly", "weeks") to coordinates in a continuous semantic space, they suffer from one fatal flaw in standard Transformer architectures:

Transformers are permutation-invariant: Because Self-Attention calculates relationships between tokens in parallel (unlike sequential models like RNNs or LSTMs), the model sees a sequence of tokens as an unordered bag of words.
Order changes meaning: The sentences "The dog chased the cat" and "The cat chased the dog" contain the exact same token IDs. Without positional information, a Transformer would compute identical mathematical representations for both sentences.
Positional Encoding fixes this by adding a unique vector to each token's embedding based on its index (position) in the sequence. This informs the model where each token resides, allowing it to distinguish order and structure.

Let's implement the standard GPT-style Trainable Absolute Positional Embedding layer and add it directly to our token embeddings.

Positional Encoding Added Successfully!
The code executed correctly. Here is exactly what we accomplished in this step:

Position IDs: We created a simple tracking tensor tensor([0, 1, 2, 3]) representing the index of each word inside our context window.
Positional Embeddings: The model mapped each of those four positions to a unique, continuous 256-dimensional vector.
Summation: By adding the token embeddings (torch.Size([4, 4, 256])) and positional embeddings (torch.Size([4, 256])), PyTorch automatically broadcasted the values to yield a final combined embedding tensor of shape torch.Size([4, 4, 256]).

In [70]:
import torch
import torch.nn as nn

# 1. Define configuration
max_length = 4  # Context window size from earlier
max_seq_len = max_length
pos_vocab_size = max_seq_len

# 2. Instantiate the Trainable Positional Embedding layer
# (Each position 0, 1, 2, 3 gets its own 256-dimensional vector)
position_embedding_layer = nn.Embedding(pos_vocab_size, embedding_dim)

# 3. Create position IDs for our batch
# We need a tensor representing [0, 1, 2, 3] for each sequence in our batch of size 4
positions = torch.arange(max_seq_len, dtype=torch.long)
print(f"Position IDs: {positions}")

# 4. Extract positional embeddings
pos_embeddings = position_embedding_layer(positions)  # Shape: (sequence_length, embedding_dim)
print(f"Positional Embeddings shape: {pos_embeddings.shape}\n")

# 5. Add Token Embeddings + Positional Embeddings
# PyTorch automatically broadcasts pos_embeddings (shape [4, 256]) to match batch_embeddings_x (shape [4, 4, 256])
final_embeddings_x = batch_embeddings_x + pos_embeddings

print(f"Input Token Embeddings shape:  {batch_embeddings_x.shape}")
print(f"Final combined Embeddings shape: {final_embeddings_x.shape}")

Position IDs: tensor([0, 1, 2, 3])
Positional Embeddings shape: torch.Size([4, 256])

Input Token Embeddings shape:  torch.Size([4, 4, 256])
Final combined Embeddings shape: torch.Size([4, 4, 256])


### Stage 2: Attention Mechanism

Now that we have prepared our input tokens, generated embeddings, and added positional representations, the next critical phase in building our LLM from scratch is the **Attention Mechanism**.

#### What is the Attention Mechanism?
In a standard neural network, inputs are processed independently. However, in language, the meaning of a word depends heavily on its context (the other words around it). The Attention Mechanism allows the model to dynamically focus on different parts of the input sequence when processing a specific token.

* **Self-Attention**: Allows each token in a sequence to interact with and "pay attention" to every other token, calculating contextualized representations.
* **Scaled Dot-Product Attention**: The mathematical formulation that calculates relevance scores between tokens using **Queries (Q)**, **Keys (K)**, and **Values (V)**.
* **Causal/Masked Attention**: Restricts tokens from looking ahead at future tokens, ensuring the model only uses past and current context during next-token prediction.

* **Multi-head Attention**:is a core component of the Transformer neural network architecture introduced in the 2017 paper "Attention Is All You Need". It extends basic self-attention by running multiple attention mechanisms (called "heads") in paralle

### Attention Mechanism Concepts

*   Simplified attention mechanism:



# self attention with trainable weights: trainable weight metrics are crucial for good context vectors.


The code has executed successfully! Here is a breakdown of what we just achieved:

Input Vector ($X$$X$): We started with our final combined embeddings of shape torch.Size([4, 4, 256]) (representing 4 batches, a sequence length of 4 tokens, and 256 embedding dimensions).
Trainable Projections: We defined three separate linear layers (W_query, W_key, and W_value) with output dimension 16 and no bias terms.

**Computation of Q, K, V:**
Query (Q): What a token is looking for.
Key (K): What information a token contains (used for matching with Queries).
Value (V): The actual semantic content that gets aggregated based on the match between Q and K.
All three result tensors successfully match the target shape torch.Size([4, 4, 16]).

In [74]:
import torch
import torch.nn as nn

# Let's use our final combined embeddings from the previous step
# final_embeddings_x has shape: (batch_size, sequence_length, embedding_dim) -> (4, 4, 256)
x = final_embeddings_x
batch_size, seq_len, d_in = x.shape

# Define the target dimension for Q, K, V projection
d_out = 16  # projecting down to a smaller dimension for demonstration

# 1. Define trainable projection layers
W_query = nn.Linear(d_in, d_out, bias=False)
W_key = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

# 2. Project inputs to compute Queries, Keys, and Values
queries = W_query(x)  # Shape: (batch_size, seq_len, d_out)
keys = W_key(x)      # Shape: (batch_size, seq_len, d_out)
values = W_value(x)  # Shape: (batch_size, seq_len, d_out)

print(f"Input shape (X): {x.shape} (batch_size, seq_len, d_in)")
print(f"Query shape (Q): {queries.shape} (batch_size, seq_len, d_out)")
print(f"Key shape (K):   {keys.shape} (batch_size, seq_len, d_out)")
print(f"Value shape (V): {values.shape} (batch_size, seq_len, d_out)")

Input shape (X): torch.Size([4, 4, 256]) (batch_size, seq_len, d_in)
Query shape (Q): torch.Size([4, 4, 16]) (batch_size, seq_len, d_out)
Key shape (K):   torch.Size([4, 4, 16]) (batch_size, seq_len, d_out)
Value shape (V): torch.Size([4, 4, 16]) (batch_size, seq_len, d_out)


In [76]:
import math
import torch.nn.functional as F

# 1. Compute raw similarity scores using matrix multiplication
# queries: (batch_size, seq_len, d_out) -> (4, 4, 16)
# keys: (batch_size, seq_len, d_out) -> (4, 4, 16)
# We transpose the keys tensor along the sequence length and projection dimension to compute dot products
raw_scores = torch.bmm(queries, keys.transpose(1, 2))

# 2. Scale scores by the square root of the key dimension (d_out = 16, so sqrt(d_out) = 4)
d_k = keys.shape[-1]
scaled_scores = raw_scores / math.sqrt(d_k)

# 3. Apply softmax to get normalized attention weights along the last dimension (summing to 1 for each query)
attention_weights = F.softmax(scaled_scores, dim=-1)

print(f"Raw Scores shape:         {raw_scores.shape} (batch_size, seq_len, seq_len)")
print(f"Attention Weights shape:  {attention_weights.shape} (batch_size, seq_len, seq_len)")
print("\nAttention Weights for the first sequence in the batch (each row sums to 1.0):")
display(attention_weights[0])

Raw Scores shape:         torch.Size([4, 4, 4]) (batch_size, seq_len, seq_len)
Attention Weights shape:  torch.Size([4, 4, 4]) (batch_size, seq_len, seq_len)

Attention Weights for the first sequence in the batch (each row sums to 1.0):


tensor([[0.1502, 0.1584, 0.2273, 0.4641],
        [0.2535, 0.3851, 0.1422, 0.2192],
        [0.1404, 0.4633, 0.2726, 0.1237],
        [0.3006, 0.1159, 0.2597, 0.3238]], grad_fn=<SelectBackward0>)

### Why We Scale the Dot Product by $\sqrt{d_k}$

In the self-attention formula, the dot product of the Query ($Q$) and Key ($K$) is divided by the scaling factor $\sqrt{d_k}$:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right)V$$

Here is why this scaling factor is mathematically necessary:

#### 1. Preventing Large Dot Products
Assume that the components of our Query vector $q$ and Key vector $k$ are independent random variables with a mean of $0$ and variance of $1$:
$$\mathbb{E}[q_i] = 0, \quad \text{Var}(q_i) = 1$$
$$\mathbb{E}[k_i] = 0, \quad \text{Var}(k_i) = 1$$

Their dot product is the sum of $d_k$ individual products:
$$q \cdot k = \sum_{i=1}^{d_k} q_i k_i$$

By mathematical expectation and variance rules, the variance of this sum grows linearly with the dimension of the vectors ($d_k$):
$$\text{Var}(q \cdot k) = d_k$$

As the dimensionality $d_k$ becomes very large, the variance of the raw dot products becomes extremely large, causing some dot products to become extremely positive or extremely negative.

#### 2. Avoiding the Softmax Vanishing Gradient Problem
When the input values to the **Softmax** function are very large in magnitude, the softmax function behaves like a "hardmax" (assigning a probability of almost $1.0$ to the maximum value and $0.0$ to all others).

When this happens, the gradients of the softmax function with respect to its inputs become **extremely close to zero** (vanishing gradients). During training, this halts backpropagation through the attention weights, preventing the model from learning.

#### 3. How Scaling Solves This
By dividing the dot product by $\sqrt{d_k}$, we scale the variance of the dot product back down to $1$:
$$\text{Var}\left(\frac{q \cdot k}{\sqrt{d_k}}\right) = \frac{1}{d_k} \text{Var}(q \cdot k) = \frac{d_k}{d_k} = 1$$

This keeps the inputs to the softmax function in a reasonable range, ensuring stable gradients and smooth weight updates during training.